In [9]:
import gradio as gr
import cv2
import os
import numpy as np
from PIL import Image
import sqlite3
import time

# Paths and Setup
db_path = "face_recognition.db"
dataset_path = "dataset"
trainer_path = "trainer"
os.makedirs(dataset_path, exist_ok=True)
os.makedirs(trainer_path, exist_ok=True)

# Create database and tables if they don't exist
def create_db():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS users (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        name TEXT NOT NULL UNIQUE)''')
    cursor.execute('''CREATE TABLE IF NOT EXISTS images (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        user_id INTEGER,
                        image_path TEXT NOT NULL,
                        FOREIGN KEY(user_id) REFERENCES users(id))''')
    conn.commit()
    conn.close()

create_db()

# Global state to track images captured
images_captured = False

# Capture Training Data
def capture_training_data(user_name):
    global images_captured
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Check if user already exists
    cursor.execute("SELECT id FROM users WHERE name = ?", (user_name,))
    existing_user = cursor.fetchone()
    if existing_user:
        conn.close()
        return f"User '{user_name}' already exists. Please provide a different name."

    # Add user to the database
    cursor.execute("INSERT INTO users (name) VALUES (?)", (user_name,))
    user_id = cursor.lastrowid
    conn.commit()

    cap = cv2.VideoCapture(0)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    sample_count = 0
    max_samples = 10
    instructions = ["Look straight", "Turn left", "Turn right", "Tilt up", "Tilt down"]
    instruction_interval = max_samples // len(instructions)
    current_instruction = 0

    while sample_count < max_samples:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        instruction_text = instructions[current_instruction]

        cv2.putText(frame, instruction_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        for (x, y, w, h) in faces:
            face_img = gray[y:y+h, x:x+w]
            image_path = f"{dataset_path}/User.{user_id}.{int(time.time())}_{sample_count}.jpg"
            cv2.imwrite(image_path, face_img)

            # Verify image saved
            if not os.path.exists(image_path):
                print(f"Failed to save image at {image_path}. Skipping...")
                continue

            # Insert image path into the database
            cursor.execute("INSERT INTO images (user_id, image_path) VALUES (?, ?)", (user_id, image_path))
            conn.commit()

            sample_count += 1
            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        cv2.imshow("Capturing Training Data", frame)
        cv2.waitKey(1000) if sample_count % instruction_interval == 0 else cv2.waitKey(1)
        current_instruction = min(current_instruction + 1, len(instructions) - 1)

    cap.release()
    cv2.destroyAllWindows()
    images_captured = True
    conn.close()

    # Train the model after capturing images
    training_result = train_model()
    return f"Captured {sample_count} images for user '{user_name}'.\n{training_result}"

# Train Model
def train_model():
    if not images_captured:
        return "Please capture images first before training the model."

    recognizer = cv2.face.LBPHFaceRecognizer_create()
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''SELECT images.image_path, users.id FROM images
                      JOIN users ON images.user_id = users.id''')
    rows = cursor.fetchall()

    faces = []
    ids = []

    for row in rows:
        image_path, user_id = row
        if not os.path.exists(image_path):
            print(f"Skipping missing file: {image_path}")
            continue
        try:
            gray_image = Image.open(image_path).convert('L')
            image_np = np.array(gray_image, 'uint8')
            faces_detected = face_cascade.detectMultiScale(image_np)

            for (x, y, w, h) in faces_detected:
                faces.append(image_np[y:y+h, x:x+w])
                ids.append(user_id)
        except Exception as e:
            print(f"Error processing image {image_path}: {e}")

    if not faces or not ids:
        conn.close()
        return "No valid training data found. Ensure faces are properly captured."

    recognizer.train(faces, np.array(ids))
    recognizer.write(f'{trainer_path}/trainer.yml')
    conn.close()
    return "Model trained successfully!"

# Recognize Faces
def recognize_faces():
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    model_path = f'{trainer_path}/trainer.yml'

    if not os.path.exists(model_path):
        return "Model not found. Please train the model first."

    recognizer.read(model_path)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    cap = cv2.VideoCapture(0)
    confidence_threshold = 50

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        for (x, y, w, h) in faces:
            id, confidence = recognizer.predict(gray[y:y+h, x:x+w])
            confidence_percentage = 100 - confidence

            if confidence_percentage > confidence_threshold:
                cursor.execute("SELECT name FROM users WHERE id = ?", (id,))
                row = cursor.fetchone()
                name = row[0] if row else "Unknown"
                text = f"{name} ({confidence_percentage:.2f}%)"
                color = (0, 255, 0)
            else:
                text = "Unknown"
                color = (0, 0, 255)

            cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

        cv2.imshow("Face Recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    conn.close()
    return "Face recognition ended."

# Gradio Interface
def run_capture(user_name):
    return capture_training_data(user_name)

def run_training():
    return train_model()

def run_recognition():
    return recognize_faces()

with gr.Blocks(css=""" 
    html, body { background-color: #e0e0e0 !important; font-family: Arial, sans-serif; }
    .title { display: flex; justify-content: center; align-items: center; font-size: 36px; font-weight: bold; color: #333; margin-bottom: 20px; }
    .button { background-color: #4CAF50; color: white; font-size: 20px; padding: 12px 12px; border-radius: 6px; cursor: pointer; transition: 0.3s; width: 200px; margin: 10px auto; }
    .button:hover { background-color: #45a049; }
    .gradio-textbox, .gradio-output { font-size: 12px; padding: 6px; margin: 6px 0; border: 1px solid #ccc; border-radius: 6px; width: 300px; background-color: #FFFFFF; margin: 10px auto; }
    .gradio-container { max-width: 600px; margin: 20px auto; padding: 15px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15); border-radius: 12px; background-color: #43A5BE; display: flex; flex-direction: column; align-items: center; }
""") as demo:
    gr.HTML('<div class="title">Face Recognition System</div>')

    with gr.Column(elem_id="gradio-container"):
        name_input = gr.Textbox(label="Enter Name", elem_classes="gradio-textbox")
        capture_btn = gr.Button("Capture & Train", elem_classes="button")
        capture_output = gr.Textbox(placeholder="Capture and training output will appear here...", elem_classes="gradio-output")
        capture_btn.click(fn=run_capture, inputs=name_input, outputs=capture_output)

   

    with gr.Tab("Recognize Faces"):
        recognize_btn = gr.Button("Start Recognition", elem_classes="button")
        recognize_btn.click(fn=run_recognition, outputs=None)

demo.launch(debug=True)


* Running on local URL:  http://127.0.0.1:7867

To create a public link, set `share=True` in `launch()`.


C:\Users\Admin\anaconda3\Lib\site-packages\gradio\blocks.py:1748: UserWarning: A function (run_recognition) returned too many output values (needed: 0, returned: 1). Ignoring extra values.
    Output components:
        []
    Output values returned:
        ["Face recognition ended."]
  warnings.warn(


Keyboard interruption in main thread... closing server.


In [ ]:
import gradio as gr
import cv2
import os
import numpy as np
from PIL import Image
import sqlite3
import time

# Paths and Setup
db_path = "face_recognition.db"
dataset_path = "dataset"
trainer_path = "trainer"
os.makedirs(dataset_path, exist_ok=True)
os.makedirs(trainer_path, exist_ok=True)

# Create database and tables if they don't exist
def create_db():
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS users (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        name TEXT NOT NULL UNIQUE)''')
    cursor.execute('''CREATE TABLE IF NOT EXISTS images (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        user_id INTEGER,
                        image_path TEXT NOT NULL,
                        FOREIGN KEY(user_id) REFERENCES users(id))''')
    conn.commit()
    conn.close()

create_db()

# Global state to track images captured
images_captured = False

# Capture Training Data
def capture_training_data(user_name):
    global images_captured
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Check if user already exists
    cursor.execute("SELECT id FROM users WHERE name = ?", (user_name,))
    existing_user = cursor.fetchone()
    if existing_user:
        conn.close()
        return f"User '{user_name}' already exists. Please provide a different name."

    # Add user to the database
    cursor.execute("INSERT INTO users (name) VALUES (?)", (user_name,))
    user_id = cursor.lastrowid
    conn.commit()

    cap = cv2.VideoCapture(0)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    sample_count = 0
    max_samples = 10
    instructions = ["Look straight", "Turn left", "Turn right", "Tilt up", "Tilt down"]
    instruction_interval = max_samples // len(instructions)
    current_instruction = 0

    while sample_count < max_samples:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)
        instruction_text = instructions[current_instruction]

        cv2.putText(frame, instruction_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        for (x, y, w, h) in faces:
            face_img = gray[y:y+h, x:x+w]
            image_path = f"{dataset_path}/User.{user_id}.{int(time.time())}_{sample_count}.jpg"
            cv2.imwrite(image_path, face_img)

            # Verify image saved
            if not os.path.exists(image_path):
                print(f"Failed to save image at {image_path}. Skipping...")
                continue

            # Insert image path into the database
            cursor.execute("INSERT INTO images (user_id, image_path) VALUES (?, ?)", (user_id, image_path))
            conn.commit()

            sample_count += 1
            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)

        cv2.imshow("Capturing Training Data", frame)
        cv2.waitKey(1000) if sample_count % instruction_interval == 0 else cv2.waitKey(1)
        current_instruction = min(current_instruction + 1, len(instructions) - 1)

    cap.release()
    cv2.destroyAllWindows()
    images_captured = True
    conn.close()

    # Train the model after capturing images
    training_result = train_model()
    return f"Captured {sample_count} images for user '{user_name}'.\n{training_result}"

# Train Model
def train_model():
    if not images_captured:
        return "Please capture images first before training the model."

    recognizer = cv2.face.LBPHFaceRecognizer_create()
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    cursor.execute('''SELECT images.image_path, users.id FROM images
                      JOIN users ON images.user_id = users.id''')
    rows = cursor.fetchall()

    faces = []
    ids = []

    for row in rows:
        image_path, user_id = row
        if not os.path.exists(image_path):
            print(f"Skipping missing file: {image_path}")
            continue
        try:
            gray_image = Image.open(image_path).convert('L')
            image_np = np.array(gray_image, 'uint8')
            faces_detected = face_cascade.detectMultiScale(image_np)

            for (x, y, w, h) in faces_detected:
                faces.append(image_np[y:y+h, x:x+w])
                ids.append(user_id)
        except Exception as e:
            print(f"Error processing image {image_path}: {e}")

    if not faces or not ids:
        conn.close()
        return "No valid training data found. Ensure faces are properly captured."

    recognizer.train(faces, np.array(ids))
    recognizer.write(f'{trainer_path}/trainer.yml')
    conn.close()
    return "Model trained successfully!"

# Recognize Faces
def recognize_faces():
    recognizer = cv2.face.LBPHFaceRecognizer_create()
    model_path = f'{trainer_path}/trainer.yml'

    if not os.path.exists(model_path):
        return "Model not found. Please train the model first."

    recognizer.read(model_path)
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    cap = cv2.VideoCapture(0)
    confidence_threshold = 50

    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        for (x, y, w, h) in faces:
            id, confidence = recognizer.predict(gray[y:y+h, x:x+w])
            confidence_percentage = 100 - confidence

            if confidence_percentage > confidence_threshold:
                cursor.execute("SELECT name FROM users WHERE id = ?", (id,))
                row = cursor.fetchone()
                name = row[0] if row else "Unknown"
                text = f"{name} ({confidence_percentage:.2f}%)"
                color = (0, 255, 0)
            else:
                text = "Unknown"
                color = (0, 0, 255)

            cv2.putText(frame, text, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
            cv2.rectangle(frame, (x, y), (x+w, y+h), color, 2)

        cv2.imshow("Face Recognition", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()
    conn.close()
    return None  # No output needed

# Gradio Interface
def run_capture(user_name):
    return capture_training_data(user_name)

def run_recognition():
    return recognize_faces()

with gr.Blocks(css=""" 
    html, body { background-color: #e0e0e0 !important; font-family: Arial, sans-serif; }
    .title { display: flex; justify-content: center; align-items: center; font-size: 36px; font-weight: bold; color: #333; margin-bottom: 20px; }
    .button { background-color: #4CAF50; color: white; font-size: 20px; padding: 12px 12px; border-radius: 6px; cursor: pointer; transition: 0.3s; width: 200px; margin: 10px auto; }
    .button:hover { background-color: #45a049; }
    .gradio-textbox, .gradio-output { font-size: 12px; padding: 6px; margin: 6px 0; border: 1px solid #ccc; border-radius: 6px; width: 300px; background-color: #FFFFFF; margin: 10px auto; }
    .gradio-container { max-width: 600px; margin: 20px auto; padding: 15px; box-shadow: 0 4px 12px rgba(0, 0, 0, 0.15); border-radius: 12px; background-color: #43A5BE; display: flex; flex-direction: column; align-items: center; }
""") as demo:
    gr.HTML('<div class="title">Face Recognition System</div>')

    with gr.Column(elem_id="gradio-container"):
        name_input = gr.Textbox(label="Enter Name", elem_classes="gradio-textbox")
        capture_btn = gr.Button("Capture & Train", elem_classes="button")
        capture_output = gr.Textbox(placeholder="Capture and training output will appear here...", elem_classes="gradio-output")
        capture_btn.click(fn=run_capture, inputs=name_input, outputs=capture_output)

    recognize_btn = gr.Button("Start Recognition", elem_classes="button")
    recognize_btn.click(fn=run_recognition, outputs=None)

demo.launch(debug=True)


* Running on local URL:  http://127.0.0.1:7867

To create a public link, set `share=True` in `launch()`.
